In [ ]:
# Install atlasopenmagic. --user is only required on SWAN, but shouldn't cause issues elsewhere.
%pip install --user atlasopenmagic

In [ ]:
# SWAN does not include the local package installation in the PYTHONPATH, so we work around this.
# Again, this shouldn't cause issues elsewhere.
import sys
import os
sys.path += [ f'{os.environ["HOME"]}/.local/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages' ]

# We can now safely import atlasopenmagic and install its recommended packages
import atlasopenmagic
atlasopenmagic.install_from_environment()

In [ ]:
# File reading libraries
import uproot
import atlasopenmagic as atom
# Analysis libraries
import awkward as ak
import numpy as np
# Utility libraries
import matplotlib.pyplot as plt
import matplotlib
# Change our plotting font to sans serif, for aesthetics!
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica","Arial","Nimbus Sans","DejaVu Sans"],
    "svg.fonttype": "path"
})
from tqdm import tqdm
import time

In [ ]:
# - Useful plotting functions - #
# Use innerward facing ticks in all direction
def inner_ticks(ax:matplotlib.axes.Axes):
    ax.tick_params(direction="in",which="both",top=True,right=True)

# Plot an ATLAS Open Data label on a pair of axes
def atlas_label(ax:matplotlib.axes.Axes,loc="upper right",text="Open Data",fontsize=16,pos_ovr=None):
    locs = {
        "upper left" : (.04, .88),
        "upper right": (.96, .88),
    }
    if (pos_ovr is not None) and isinstance(pos_ovr,tuple):
        pos = pos_ovr 
    else: pos = locs[loc]
    align = ("left","right","center"); ha = "left"
    for a in align:
        if a in loc: ha = a
    ax.text(
        *pos,r"$\mathbfit{ATLAS}$ "+text,
        transform=ax.transAxes,fontsize=fontsize,ha=ha,va="bottom"
    )

In [ ]:
# Fetch minimum bias heavy ion collision data
atom.set_release("2024r-hi")
sample_defs = {
    "Data": {'dids': ["data"], 'color': 'red'}
}
data_samples = atom.build_dataset(sample_defs,skim="noskim",protocol="https",cache=True)

In [ ]:
# Histogram parameters for the FCal total transverse energy histogram which we want to fill
h_def = {
    "bin_width": 20e3, # MeV
    "range": (0,5e6) # MeV
}
h_def["n_bins"] = int((h_def["range"][1] - h_def["range"][0]) / h_def["bin_width"])
# Initialise an empty total ET histogram
h_total_eT, h_def["bin_edges"] = np.histogram([-1],h_def["n_bins"],h_def["range"])

In [ ]:
# Take a url and grab the total transverse energy histogram from its ROOT file
def grab_hist_from_file(url:str)->np.ndarray:
    with uproot.open(f"{url}:CollectionTree") as tree:
        fcal = ak.zip({
        branch.split(".")[1]: tree[branch].array() for branch in (
            f"EventInfoAuxDyn.FCalEt{x}" for x in ("A","C")
        )})
        fcal["TotalEt"] = fcal["FCalEtA"] + fcal["FCalEtC"]
        this_hist,_ = np.histogram(fcal["TotalEt"],bins=h_def["n_bins"],range=h_def["range"])
    return this_hist

# Converts a numpy histogram into a sparkline histogram
# Allows us to watch our histogram fill in real time!
blocks = " ▁▂▃▄▅▆▇█"
def to_sparkline(hist:np.ndarray,lim=20)->str:
    if len(hist) > lim: # Trim the histogram if it is too large, taking evenly-spaced points along it
        hist = np.array([hist[int( (len(hist)-1)*i/lim )] for i in range(lim)])
    if np.max(hist)==0:
        return blocks[0]*len(hist)
    norm = hist / np.max(hist)
    return f'{"".join(blocks[int(v * (len(blocks) - 1))] for v in norm)} Events: {int(np.sum(hist))}'

In [ ]:
start = time.time()
print("Processing datasets...")
pbar = tqdm(data_samples["Data"]["list"][:5],desc=to_sparkline(h_total_eT))
for url in pbar:
    h_total_eT = h_total_eT + grab_hist_from_file(url)
    # Update progress bar histogram
    pbar.set_description(to_sparkline(h_total_eT))
end = time.time()
print(f"Total time: {round((end_all - start_all) / 60, 1)} mins")